### Setup and load raw data

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

race_raw = pd.read_csv(RAW / "race_results.csv")
quali_raw = pd.read_csv(RAW / "qualifying_results.csv")

print("Project root:", PROJECT_ROOT)
print("Race raw:", race_raw.shape)
print("Qualifying raw:", quali_raw.shape)

race_raw.head()

Project root: /Users/lukeweeklund/F1 Project
Race raw: (6911, 37)
Qualifying raw: (6891, 28)


,number,position,positionText,points,grid,laps,status,driverId,driverNumber,driverCode,...,round,raceName,date,circuitId,circuitName,locality,country,lat,long,data_type
0,8,1,1,25.0,3,49,Finished,alonso,14.0,ALO,...,1,Bahrain Grand Prix,NaN,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,race_results
1,7,2,2,18.0,2,49,Finished,massa,19.0,MAS,...,1,Bahrain Grand Prix,NaN,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,race_results
2,2,3,3,15.0,4,49,Finished,hamilton,44.0,HAM,...,1,Bahrain Grand Prix,NaN,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,race_results
3,5,4,4,12.0,1,49,Finished,vettel,5.0,VET,...,1,Bahrain Grand Prix,NaN,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,race_results
4,4,5,5,10.0,5,49,Finished,rosberg,6.0,ROS,...,1,Bahrain Grand Prix,NaN,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,race_results


### Confirm required raw columns

In [2]:
required_race_cols = [
    "season", "round", "raceName", "circuitId", "circuitName",
    "locality", "country", "lat", "long",
    "driverId", "givenName", "familyName", "driverNationality",
    "constructorId", "constructorName", "constructorNationality",
    "position", "grid", "points", "laps", "status"
]

required_quali_cols = [
    "season", "round", "driverId", "position"
]

missing_race_cols = [col for col in required_race_cols if col not in race_raw.columns]
missing_quali_cols = [col for col in required_quali_cols if col not in quali_raw.columns]

print("Missing race columns:", missing_race_cols)
print("Missing qualifying columns:", missing_quali_cols)

if missing_race_cols:
    raise KeyError(f"Missing race columns: {missing_race_cols}")

if missing_quali_cols:
    raise KeyError(f"Missing qualifying columns: {missing_quali_cols}")

Missing race columns: []
Missing qualifying columns: []


### Clean race results

In [3]:
race = race_raw[required_race_cols].copy()

race = race.rename(columns={
    "raceName": "race_name",
    "circuitId": "circuit_id",
    "circuitName": "circuit_name",
    "locality": "circuit_locality",
    "country": "circuit_country",
    "lat": "circuit_lat",
    "long": "circuit_long",
    "driverId": "driver_id",
    "givenName": "given_name",
    "familyName": "family_name",
    "driverNationality": "driver_nationality",
    "constructorId": "constructor_id",
    "constructorName": "constructor_name",
    "constructorNationality": "constructor_nationality",
    "position": "finish_position",
    "grid": "grid_position"
})

# Convert core numeric fields safely
numeric_cols = [
    "season", "round", "finish_position", "grid_position",
    "points", "laps", "circuit_lat", "circuit_long"
]

for col in numeric_cols:
    race[col] = pd.to_numeric(race[col], errors="coerce")

# Keep only rows that can identify one driver-race result
race = race.dropna(subset=[
    "season", "round", "driver_id", "constructor_id", "finish_position"
]).copy()

race["season"] = race["season"].astype(int)
race["round"] = race["round"].astype(int)
race["finish_position"] = race["finish_position"].astype(int)
race["grid_position"] = race["grid_position"].fillna(0).astype(int)
race["points"] = race["points"].fillna(0)
race["laps"] = race["laps"].fillna(0).astype(int)

race["driver_name"] = (
    race["given_name"].astype(str).str.strip()
    + " "
    + race["family_name"].astype(str).str.strip()
)

# ------------------------------------------------------------------------------
# CREATE STABLE IDENTIFIERS
# These IDs are used throughout the project to simplify joins and groupby logic.
# ------------------------------------------------------------------------------

# Unique race
race["race_id"] = (
    race["season"].astype(str)
    + "_R"
    + race["round"].astype(str)
)

# Unique driver entry within a race (should be unique for every row)
race["driver_race_id"] = (
    race["race_id"]
    + "_"
    + race["driver_id"]
)

# Unique constructor entry within a race
race["constructor_race_id"] = (
    race["race_id"]
    + "_"
    + race["constructor_id"]
)

# One driver's stint with one constructor across all seasons
race["driver_constructor_stint"] = (
    race["driver_id"]
    + "_"
    + race["constructor_id"]
)

# Prevent accidental duplicate driver-race rows from raw data
race = race.drop_duplicates(
    subset=["season", "round", "driver_id"],
    keep="last"
).reset_index(drop=True)

print("Clean race:", race.shape)
print("Unique races:", race[["season", "round"]].drop_duplicates().shape[0])
print("Duplicate driver-race rows:", race.duplicated(["season", "round", "driver_id"]).sum())

race.head()

Clean race: (6911, 26)
Unique races: 329
Duplicate driver-race rows: 0


,season,round,race_name,circuit_id,circuit_name,circuit_locality,circuit_country,circuit_lat,circuit_long,driver_id,...,finish_position,grid_position,points,laps,status,driver_name,race_id,driver_race_id,constructor_race_id,driver_constructor_stint
0,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,alonso,...,1,3,25.0,49,Finished,Fernando Alonso,2010_R1,2010_R1_alonso,2010_R1_ferrari,alonso_ferrari
1,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,massa,...,2,2,18.0,49,Finished,Felipe Massa,2010_R1,2010_R1_massa,2010_R1_ferrari,massa_ferrari
2,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,hamilton,...,3,4,15.0,49,Finished,Lewis Hamilton,2010_R1,2010_R1_hamilton,2010_R1_mclaren,hamilton_mclaren
3,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,vettel,...,4,1,12.0,49,Finished,Sebastian Vettel,2010_R1,2010_R1_vettel,2010_R1_red_bull,vettel_red_bull
4,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,rosberg,...,5,5,10.0,49,Finished,Nico Rosberg,2010_R1,2010_R1_rosberg,2010_R1_mercedes,rosberg_mercedes


### Clean qualifying results

In [4]:
quali = quali_raw[required_quali_cols].copy()

quali = quali.rename(columns={
    "driverId": "driver_id",
    "position": "qualifying_position"
})

# Convert qualifying fields safely
for col in ["season", "round", "qualifying_position"]:
    quali[col] = pd.to_numeric(quali[col], errors="coerce")

quali = quali.dropna(subset=[
    "season", "round", "driver_id", "qualifying_position"
]).copy()

quali["season"] = quali["season"].astype(int)
quali["round"] = quali["round"].astype(int)
quali["qualifying_position"] = quali["qualifying_position"].astype(int)

# Prevent duplicate qualifying rows
quali = quali.drop_duplicates(
    subset=["season", "round", "driver_id"],
    keep="last"
).reset_index(drop=True)

print("Clean qualifying:", quali.shape)
print("Unique qualifying races:", quali[["season", "round"]].drop_duplicates().shape[0])
print("Duplicate qualifying rows:", quali.duplicated(["season", "round", "driver_id"]).sum())

quali.head()

Clean qualifying: (6891, 4)
Unique qualifying races: 329
Duplicate qualifying rows: 0


,season,round,driver_id,qualifying_position
0,2010,1,vettel,1
1,2010,1,massa,2
2,2010,1,alonso,3
3,2010,1,hamilton,4
4,2010,1,rosberg,5


### Merge quali into race results

In [5]:
rows_before_quali_merge = len(race)

race = race.merge(
    quali,
    on=["season", "round", "driver_id"],
    how="left"
)

print("Rows before qualifying merge:", rows_before_quali_merge)
print("Rows after qualifying merge:", len(race))
print("Duplicate driver-race rows:", race.duplicated(["season", "round", "driver_id"]).sum())
print("Missing qualifying positions:", race["qualifying_position"].isna().sum())
print("Qualifying coverage %:", round(100 * race["qualifying_position"].notna().mean(), 2))

race.head()

Rows before qualifying merge: 6911
Rows after qualifying merge: 6911
Duplicate driver-race rows: 0
Missing qualifying positions: 25
Qualifying coverage %: 99.64


,season,round,race_name,circuit_id,circuit_name,circuit_locality,circuit_country,circuit_lat,circuit_long,driver_id,...,grid_position,points,laps,status,driver_name,race_id,driver_race_id,constructor_race_id,driver_constructor_stint,qualifying_position
0,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,alonso,...,3,25.0,49,Finished,Fernando Alonso,2010_R1,2010_R1_alonso,2010_R1_ferrari,alonso_ferrari,3.0
1,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,massa,...,2,18.0,49,Finished,Felipe Massa,2010_R1,2010_R1_massa,2010_R1_ferrari,massa_ferrari,2.0
2,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,hamilton,...,4,15.0,49,Finished,Lewis Hamilton,2010_R1,2010_R1_hamilton,2010_R1_mclaren,hamilton_mclaren,4.0
3,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,vettel,...,1,12.0,49,Finished,Sebastian Vettel,2010_R1,2010_R1_vettel,2010_R1_red_bull,vettel_red_bull,1.0
4,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,rosberg,...,5,10.0,49,Finished,Nico Rosberg,2010_R1,2010_R1_rosberg,2010_R1_mercedes,rosberg_mercedes,5.0


### Create basic race features and DNF flags

In [6]:
race["positions_gained"] = race["grid_position"] - race["finish_position"]

def classify_status(status):
    status = str(status).lower().strip()

    if status == "finished" or status.startswith("+") or "lapped" in status:
        return "Finished"

    mechanical_terms = [
        "engine", "gearbox", "hydraulic", "electrical", "electronics",
        "transmission", "fuel", "oil", "brakes", "clutch",
        "driveshaft", "suspension", "wheel", "cooling", "power unit",
        "mechanical", "radiator", "water", "water pump", "exhaust",
        "turbo", "battery", "ers", "throttle", "power loss",
        "steering", "differential", "technical", "pneumatics",
        "overheating", "launch control"
    ]

    if any(term in status for term in mechanical_terms):
        return "Mechanical"

    crash_terms = [
        "accident", "collision", "spun", "damage",
        "front wing", "broken wing", "puncture", "tyre", "stalled"
    ]

    if any(term in status for term in crash_terms):
        return "Crash"

    return "Other"

race["dnf_type"] = race["status"].apply(classify_status)

race["finished"] = (race["dnf_type"] == "Finished").astype(int)
race["mechanical_dnf"] = (race["dnf_type"] == "Mechanical").astype(int)
race["crash_dnf"] = (race["dnf_type"] == "Crash").astype(int)
race["other_dnf"] = (race["dnf_type"] == "Other").astype(int)

print("DNF type counts:")
print(race["dnf_type"].value_counts())

race[["status", "dnf_type", "finished", "mechanical_dnf", "crash_dnf", "other_dnf"]].head(20)

DNF type counts:
dnf_type
Finished      5699
Mechanical     512
Crash          423
Other          277
Name: count, dtype: int64


,status,dnf_type,finished,mechanical_dnf,crash_dnf,other_dnf
0,Finished,Finished,1,0,0,0
1,Finished,Finished,1,0,0,0
2,Finished,Finished,1,0,0,0
3,Finished,Finished,1,0,0,0
4,Finished,Finished,1,0,0,0
5,Finished,Finished,1,0,0,0
6,Finished,Finished,1,0,0,0
7,Finished,Finished,1,0,0,0
8,Finished,Finished,1,0,0,0
9,Finished,Finished,1,0,0,0


### Build teammate lookup table

In [7]:
teammate = race[[
    "season", "round", "constructor_id",
    "driver_id", "driver_name",
    "finish_position", "qualifying_position",
    "grid_position", "points"
]].copy()

teammate = teammate.rename(columns={
    "driver_id": "teammate_id",
    "driver_name": "teammate_name",
    "finish_position": "teammate_finish",
    "qualifying_position": "teammate_qualifying",
    "grid_position": "teammate_grid",
    "points": "teammate_points"
})

print("Teammate lookup:", teammate.shape)
teammate.head()

Teammate lookup: (6911, 9)


,season,round,constructor_id,teammate_id,teammate_name,teammate_finish,teammate_qualifying,teammate_grid,teammate_points
0,2010,1,ferrari,alonso,Fernando Alonso,1,3.0,3,25.0
1,2010,1,ferrari,massa,Felipe Massa,2,2.0,2,18.0
2,2010,1,mclaren,hamilton,Lewis Hamilton,3,4.0,4,15.0
3,2010,1,red_bull,vettel,Sebastian Vettel,4,1.0,1,12.0
4,2010,1,mercedes,rosberg,Nico Rosberg,5,5.0,5,10.0


### Merge teammate info safely

In [8]:
rows_before_teammate_merge = len(race)

race = race.merge(
    teammate,
    on=["season", "round", "constructor_id"],
    how="left"
)

# Remove self-matches while preserving one row per driver-race
self_match = race["driver_id"] == race["teammate_id"]

teammate_cols = [
    "teammate_id", "teammate_name", "teammate_finish",
    "teammate_qualifying", "teammate_grid", "teammate_points"
]

race.loc[self_match, teammate_cols] = np.nan

# Keep teammate row when it exists; otherwise keep the no-teammate row
race = race.sort_values(
    ["season", "round", "constructor_id", "driver_id", "teammate_id"],
    na_position="last"
)

race = race.drop_duplicates(
    subset=["season", "round", "driver_id"],
    keep="first"
).reset_index(drop=True)

print("Rows before teammate merge:", rows_before_teammate_merge)
print("Rows after teammate merge:", len(race))
print("Duplicate driver-race rows:", race.duplicated(["season", "round", "driver_id"]).sum())
print("Rows without teammate:", race["teammate_id"].isna().sum())

race[[
    "season", "round", "constructor_name",
    "driver_name", "teammate_name",
    "finish_position", "teammate_finish"
]].head(20)

Rows before teammate merge: 6911
Rows after teammate merge: 6911
Duplicate driver-race rows: 0
Rows without teammate: 3


,season,round,constructor_name,driver_name,teammate_name,finish_position,teammate_finish
0,2010,1,Ferrari,Fernando Alonso,Felipe Massa,1,2.0
1,2010,1,Ferrari,Felipe Massa,Fernando Alonso,2,1.0
2,2010,1,Force India,Vitantonio Liuzzi,Adrian Sutil,9,12.0
3,2010,1,Force India,Adrian Sutil,Vitantonio Liuzzi,12,9.0
4,2010,1,HRT,Bruno Senna,Karun Chandhok,19,24.0
5,2010,1,HRT,Karun Chandhok,Bruno Senna,24,19.0
6,2010,1,Lotus,Heikki Kovalainen,Jarno Trulli,15,17.0
7,2010,1,Lotus,Jarno Trulli,Heikki Kovalainen,17,15.0
8,2010,1,McLaren,Jenson Button,Lewis Hamilton,7,3.0
9,2010,1,McLaren,Lewis Hamilton,Jenson Button,3,7.0


### Create teammate comparison features

In [9]:
race["beat_teammate"] = np.where(
    race["teammate_finish"].notna(),
    (race["finish_position"] < race["teammate_finish"]).astype(int),
    np.nan
)

race["finish_gap"] = np.where(
    race["teammate_finish"].notna(),
    race["teammate_finish"] - race["finish_position"],
    np.nan
)

race["beat_teammate_quali"] = np.where(
    race["qualifying_position"].notna() & race["teammate_qualifying"].notna(),
    (race["qualifying_position"] < race["teammate_qualifying"]).astype(int),
    np.nan
)

race["qualifying_gap"] = np.where(
    race["qualifying_position"].notna() & race["teammate_qualifying"].notna(),
    race["teammate_qualifying"] - race["qualifying_position"],
    np.nan
)

race["beat_teammate_grid"] = np.where(
    race["teammate_grid"].notna(),
    (race["grid_position"] < race["teammate_grid"]).astype(int),
    np.nan
)

race["grid_gap"] = np.where(
    race["teammate_grid"].notna(),
    race["teammate_grid"] - race["grid_position"],
    np.nan
)

race["points_gap"] = np.where(
    race["teammate_points"].notna(),
    race["points"] - race["teammate_points"],
    np.nan
)

race[[
    "driver_name", "teammate_name",
    "finish_position", "teammate_finish", "beat_teammate", "finish_gap",
    "qualifying_position", "teammate_qualifying", "beat_teammate_quali", "qualifying_gap",
    "grid_position", "teammate_grid", "beat_teammate_grid", "grid_gap",
    "points", "teammate_points", "points_gap"
]].head(20)

,driver_name,teammate_name,finish_position,teammate_finish,beat_teammate,finish_gap,qualifying_position,teammate_qualifying,beat_teammate_quali,qualifying_gap,grid_position,teammate_grid,beat_teammate_grid,grid_gap,points,teammate_points,points_gap
0,Fernando Alonso,Felipe Massa,1,2.0,1.0,1.0,3.0,2.0,0.0,-1.0,3,2.0,0.0,-1.0,25.0,18.0,7.0
1,Felipe Massa,Fernando Alonso,2,1.0,0.0,-1.0,2.0,3.0,1.0,1.0,2,3.0,1.0,1.0,18.0,25.0,-7.0
2,Vitantonio Liuzzi,Adrian Sutil,9,12.0,1.0,3.0,12.0,10.0,0.0,-2.0,12,10.0,0.0,-2.0,2.0,0.0,2.0
3,Adrian Sutil,Vitantonio Liuzzi,12,9.0,0.0,-3.0,10.0,12.0,1.0,2.0,10,12.0,1.0,2.0,0.0,2.0,-2.0
4,Bruno Senna,Karun Chandhok,19,24.0,1.0,5.0,23.0,24.0,1.0,1.0,23,24.0,1.0,1.0,0.0,0.0,0.0
5,Karun Chandhok,Bruno Senna,24,19.0,0.0,-5.0,24.0,23.0,0.0,-1.0,24,23.0,0.0,-1.0,0.0,0.0,0.0
6,Heikki Kovalainen,Jarno Trulli,15,17.0,1.0,2.0,21.0,20.0,0.0,-1.0,21,20.0,0.0,-1.0,0.0,0.0,0.0
7,Jarno Trulli,Heikki Kovalainen,17,15.0,0.0,-2.0,20.0,21.0,1.0,1.0,20,21.0,1.0,1.0,0.0,0.0,0.0
8,Jenson Button,Lewis Hamilton,7,3.0,0.0,-4.0,8.0,4.0,0.0,-4.0,8,4.0,0.0,-4.0,6.0,15.0,-9.0
9,Lewis Hamilton,Jenson Button,3,7.0,1.0,4.0,4.0,8.0,1.0,4.0,4,8.0,1.0,4.0,15.0,6.0,9.0


### Sort final modeling dataset

In [10]:
race = race.sort_values(
    ["season", "round", "finish_position"]
).reset_index(drop=True)

print("Final modeling dataset v1:", race.shape)
race.head()

Final modeling dataset v1: (6911, 46)


,season,round,race_name,circuit_id,circuit_name,circuit_locality,circuit_country,circuit_lat,circuit_long,driver_id,...,teammate_qualifying,teammate_grid,teammate_points,beat_teammate,finish_gap,beat_teammate_quali,qualifying_gap,beat_teammate_grid,grid_gap,points_gap
0,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,alonso,...,2.0,2.0,18.0,1.0,1.0,0.0,-1.0,0.0,-1.0,7.0
1,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,massa,...,3.0,3.0,25.0,0.0,-1.0,1.0,1.0,1.0,1.0,-7.0
2,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,hamilton,...,8.0,8.0,6.0,1.0,4.0,1.0,4.0,1.0,4.0,9.0
3,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,vettel,...,6.0,6.0,4.0,1.0,4.0,1.0,5.0,1.0,5.0,8.0
4,2010,1,Bahrain Grand Prix,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,rosberg,...,7.0,7.0,8.0,1.0,1.0,1.0,2.0,1.0,2.0,2.0


### Technical validation

In [11]:
print("=" * 80)
print("VALIDATION — MODELING DATASET V1")
print("=" * 80)

print("\nDATASET SHAPE")
print("-" * 80)
print("Rows:", len(race))
print("Columns:", race.shape[1])
print("Unique races:", race[["season", "round"]].drop_duplicates().shape[0])
print("Unique drivers:", race["driver_id"].nunique())
print("Unique constructors:", race["constructor_id"].nunique())

print("\nROW INTEGRITY")
print("-" * 80)
duplicate_driver_races = race.duplicated(["season", "round", "driver_id"]).sum()
print("Duplicate driver-race pairs:", duplicate_driver_races)
print("Missing driver IDs:", race["driver_id"].isna().sum())
print("Missing constructor IDs:", race["constructor_id"].isna().sum())

print("\nQUALIFYING COVERAGE")
print("-" * 80)
print("Rows missing qualifying:", race["qualifying_position"].isna().sum())
print("Qualifying coverage %:", round(100 * race["qualifying_position"].notna().mean(), 2))

print("\nTEAMMATE FEATURE CHECK")
print("-" * 80)
print("Rows without teammate:", race["teammate_id"].isna().sum())
print("Rows with teammate but missing teammate finish:",
      race[(race["teammate_id"].notna()) & (race["teammate_finish"].isna())].shape[0])
print("Rows with no teammate but beat_teammate filled:",
      race[(race["teammate_id"].isna()) & (race["beat_teammate"].notna())].shape[0])
print("Rows with no teammate but beat_teammate_grid filled:",
      race[(race["teammate_id"].isna()) & (race["beat_teammate_grid"].notna())].shape[0])
print("Rows with no teammate but points_gap filled:",
      race[(race["teammate_id"].isna()) & (race["points_gap"].notna())].shape[0])

print("\nTEAM SIZE DISTRIBUTION")
print("-" * 80)
team_size_dist = (
    race.groupby(["season", "round", "constructor_id"])
    .size()
    .value_counts()
    .sort_index()
)
print(team_size_dist)

print("\nDNF TYPE COUNTS")
print("-" * 80)
print(race["dnf_type"].value_counts(dropna=False))

print("\nVALUE RANGE CHECKS")
print("-" * 80)
print("Finish position min/max:", race["finish_position"].min(), race["finish_position"].max())
print("Grid position min/max:", race["grid_position"].min(), race["grid_position"].max())
print("Points min/max:", race["points"].min(), race["points"].max())
print("Laps min/max:", race["laps"].min(), race["laps"].max())
print("Positions gained min/max:", race["positions_gained"].min(), race["positions_gained"].max())

issues = []

if duplicate_driver_races != 0:
    issues.append("Duplicate driver-race rows exist.")

if race["driver_id"].isna().sum() != 0:
    issues.append("Missing driver IDs exist.")

if race["constructor_id"].isna().sum() != 0:
    issues.append("Missing constructor IDs exist.")

if race[(race["teammate_id"].isna()) & (race["beat_teammate"].notna())].shape[0] != 0:
    issues.append("Rows without teammates have teammate result flags.")

if race[(race["teammate_id"].isna()) & (race["beat_teammate_grid"].notna())].shape[0] != 0:
    issues.append("Rows without teammates have grid teammate flags.")

if race[(race["teammate_id"].isna()) & (race["points_gap"].notna())].shape[0] != 0:
    issues.append("Rows without teammates have points gap values.")

if issues:
    print("\nVERDICT: REVIEW NEEDED")
    for issue in issues:
        print("-", issue)
else:
    print("\nVERDICT: PASS — modeling_dataset_v1 is technically valid.")

VALIDATION — MODELING DATASET V1

DATASET SHAPE
--------------------------------------------------------------------------------
Rows: 6911
Columns: 46
Unique races: 329
Unique drivers: 83
Unique constructors: 23

ROW INTEGRITY
--------------------------------------------------------------------------------
Duplicate driver-race pairs: 0
Missing driver IDs: 0
Missing constructor IDs: 0

QUALIFYING COVERAGE
--------------------------------------------------------------------------------
Rows missing qualifying: 25
Qualifying coverage %: 99.64

TEAMMATE FEATURE CHECK
--------------------------------------------------------------------------------
Rows without teammate: 3
Rows with teammate but missing teammate finish: 0
Rows with no teammate but beat_teammate filled: 0
Rows with no teammate but beat_teammate_grid filled: 0
Rows with no teammate but points_gap filled: 0

TEAM SIZE DISTRIBUTION
--------------------------------------------------------------------------------
1       3
2    

### Analytical supercheck

In [12]:
print("=" * 80)
print("SUPERCHECK — MODELING DATASET V1")
print("=" * 80)

print("\n1. SEASON COVERAGE")
print("-" * 80)
season_summary = (
    race.groupby("season")
    .agg(
        races=("round", "nunique"),
        driver_race_rows=("driver_id", "count"),
        drivers=("driver_id", "nunique"),
        constructors=("constructor_id", "nunique")
    )
)
display(season_summary)

print("\n2. TOP POINTS RESULTS")
print("-" * 80)
display(
    race.sort_values("points", ascending=False)
    [["season", "round", "race_name", "driver_name", "constructor_name", "finish_position", "points"]]
    .head(20)
)

print("\n3. BIGGEST POSITION GAINS")
print("-" * 80)
display(
    race.sort_values("positions_gained", ascending=False)
    [["season", "round", "race_name", "driver_name", "constructor_name", "grid_position", "finish_position", "positions_gained"]]
    .head(20)
)

print("\n4. BIGGEST TEAMMATE FINISH GAPS")
print("-" * 80)
display(
    race.dropna(subset=["finish_gap"])
    .sort_values("finish_gap", ascending=False)
    [["season", "round", "race_name", "constructor_name", "driver_name", "teammate_name", "finish_position", "teammate_finish", "finish_gap"]]
    .head(20)
)

print("\n5. DRIVER TEAMMATE WIN RATE SAMPLE")
print("-" * 80)
driver_teammate_sample = (
    race.dropna(subset=["beat_teammate"])
    .groupby(["driver_id", "driver_name"], as_index=False)
    .agg(
        races_with_teammate=("beat_teammate", "count"),
        teammate_win_rate=("beat_teammate", "mean"),
        avg_finish_gap=("finish_gap", "mean"),
        avg_points_gap=("points_gap", "mean")
    )
    .query("races_with_teammate >= 20")
    .sort_values("teammate_win_rate", ascending=False)
)

display(driver_teammate_sample.head(25))

print("\n6. KNOWN DRIVER CHECKS")
print("-" * 80)
known_drivers = [
    "Lewis Hamilton",
    "Max Verstappen",
    "Fernando Alonso",
    "Sebastian Vettel",
    "Charles Leclerc",
    "Lando Norris"
]

display(
    race[race["driver_name"].isin(known_drivers)]
    .groupby("driver_name")
    .agg(
        races=("round", "count"),
        avg_finish=("finish_position", "mean"),
        avg_points=("points", "mean"),
        teammate_win_rate=("beat_teammate", "mean"),
        qualifying_teammate_win_rate=("beat_teammate_quali", "mean")
    )
    .sort_values("races", ascending=False)
)

print("\n7. SUPERCHECK VERDICT")
print("-" * 80)

supercheck_issues = []

if race["season"].min() != 2010:
    supercheck_issues.append("Minimum season is not 2010.")

if race["season"].max() != 2025:
    supercheck_issues.append("Maximum season is not 2025.")

if race[["season", "round"]].drop_duplicates().shape[0] != 329:
    supercheck_issues.append("Unique race count is not 329.")

if race["finish_position"].min() < 1:
    supercheck_issues.append("Finish position below 1 found.")

if race["grid_position"].min() < 0:
    supercheck_issues.append("Grid position below 0 found.")

if race["points"].min() < 0:
    supercheck_issues.append("Negative points found.")

if supercheck_issues:
    print("REVIEW NEEDED")
    for issue in supercheck_issues:
        print("-", issue)
else:
    print("PASS — modeling_dataset_v1 looks analytically reasonable.")

SUPERCHECK — MODELING DATASET V1

1. SEASON COVERAGE
--------------------------------------------------------------------------------


,races,driver_race_rows,drivers,constructors
season,,,,
2010,19,456,27,12
2011,19,454,28,12
2012,20,478,25,12
2013,19,418,23,11
2014,19,407,24,11
2015,19,378,22,10
2016,21,462,24,11
2017,20,400,25,10
2018,21,420,20,10



2. TOP POINTS RESULTS
--------------------------------------------------------------------------------


,season,round,race_name,driver_name,constructor_name,finish_position,points
2193,2014,19,Abu Dhabi Grand Prix,Lewis Hamilton,Mercedes,1,50.0
2194,2014,19,Abu Dhabi Grand Prix,Felipe Massa,Williams,2,36.0
2195,2014,19,Abu Dhabi Grand Prix,Valtteri Bottas,Williams,3,30.0
4753,2021,7,French Grand Prix,Max Verstappen,Red Bull,1,26.0
4473,2020,10,Russian Grand Prix,Valtteri Bottas,Mercedes,1,26.0
5833,2023,17,Qatar Grand Prix,Max Verstappen,Red Bull,1,26.0
4173,2019,16,Russian Grand Prix,Lewis Hamilton,Mercedes,1,26.0
5353,2022,15,Dutch Grand Prix,Max Verstappen,Red Bull,1,26.0
5333,2022,14,Belgian Grand Prix,Max Verstappen,Red Bull,1,26.0
5933,2023,22,Abu Dhabi Grand Prix,Max Verstappen,Red Bull,1,26.0



3. BIGGEST POSITION GAINS
--------------------------------------------------------------------------------


...


5. DRIVER TEAMMATE WIN RATE SAMPLE
--------------------------------------------------------------------------------


,driver_id,driver_name,races_with_teammate,teammate_win_rate,avg_finish_gap,avg_points_gap
48,max_verstappen,Max Verstappen,233,0.759657,2.948498,6.633047
6,barrichello,Rubens Barrichello,38,0.684211,2.736842,0.736842
52,mick_schumacher,Mick Schumacher,44,0.681818,0.340909,-0.204545
33,jules_bianchi,Jules Bianchi,34,0.676471,0.470588,0.058824
3,alonso,Fernando Alonso,287,0.672474,1.839721,3.149826
27,hadjar,Isack Hadjar,24,0.666667,1.041667,0.500000
67,russell,George Russell,152,0.644737,1.131579,0.980263
1,albon,Alexander Albon,128,0.632812,1.320312,-0.718750
54,norris,Lando Norris,152,0.631579,1.210526,1.756579
78,vettel,Sebastian Vettel,257,0.618677,1.073930,2.533074



6. KNOWN DRIVER CHECKS
--------------------------------------------------------------------------------


,races,avg_finish,avg_points,teammate_win_rate,qualifying_teammate_win_rate
driver_name,,,,,
Lewis Hamilton,328,5.076220,14.327744,0.579268,0.579268
Fernando Alonso,288,9.309028,6.260417,0.672474,0.767606
Sebastian Vettel,257,6.540856,11.568093,0.618677,0.625000
Max Verstappen,233,5.442060,14.169528,0.759657,0.801724
Charles Leclerc,173,7.445087,9.179191,0.618497,0.687861
Lando Norris,152,7.282895,8.842105,0.631579,0.677632



7. SUPERCHECK VERDICT
--------------------------------------------------------------------------------
PASS — modeling_dataset_v1 looks analytically reasonable.


### Save modeling dataset v1

In [14]:
output_path = PROCESSED / "modeling_dataset_v1.csv"

race.to_csv(output_path, index=False)

saved = pd.read_csv(output_path)

print("Saved file exists:", output_path.exists())
print("Saved path:", output_path)
print("Saved rows:", len(saved))
print("Saved columns:", saved.shape[1])
print("Saved row match:", len(saved) == len(race))
print("Saved column match:", saved.shape[1] == race.shape[1])

Saved file exists: True
Saved path: /Users/lukeweeklund/F1 Project/data/processed/modeling_dataset_v1.csv
Saved rows: 6911
Saved columns: 46
Saved row match: True
Saved column match: True
